# dbt — First Contact

dbt is a transformation tool for analytics engineering. Think of it as a framework for organizing SQL models, compiling them into runnable SQL, and executing them inside your warehouse or database. Instead of writing scattered ad hoc queries, you define named models, dependencies, tests, and documentation in a repeatable project structure.

The core dbt philosophy is ELT, not ETL. Data is first loaded into the database, and then transformed in place using SQL. In practice, a dbt model is usually just a `SELECT` statement. dbt compiles those models, resolves dependencies like `ref()` and `source()`, and runs them in the correct order. Because the logic stays in SQL and in version-controlled files, everything becomes easier to test, review, and document.

In the Citi telemetry context, the raw alerts table is engineer-friendly but not analyst-friendly. dbt transforms raw tables into clean, named, tested views that ops teams can query in Kibana or Tableau. Instead of every team reinventing alert logic, dbt gives a shared, trustworthy layer.

```text
[raw: alerts] → [dbt model: stg_alerts] → [dbt model: mart_alert_summary] → [Analyst]
```

In [1]:
DBT_PATH = "C:/py_venv/proj_educate/Scripts/dbt.exe"
import subprocess

subprocess.run(["python", "-m", "pip", "install", "dbt-postgres"], capture_output=True, text=True)
result = subprocess.run([DBT_PATH, "--version"], capture_output=True, text=True)
print(result.stdout)

Core:
  - installed: 1.11.7
  - latest:    1.11.7 - Up to date!

Plugins:
  - postgres: 1.10.0 - Up to date!





dbt init creates the project scaffold.

In [2]:
DBT_PATH = "C:/py_venv/proj_educate/Scripts/dbt.exe"
import os, subprocess
from pathlib import Path

base_dir = Path("D:/Workspace/Technologies")
project_dir = base_dir / "citi_dbt"

if not project_dir.exists():
    result = subprocess.run(
        [DBT_PATH, "init", "citi_dbt", "--skip-profile-setup"],
        capture_output=True, text=True, cwd=str(base_dir)
    )
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
else:
    print(f"Project already exists at: {project_dir}")

print("Directory contents:")
for f in sorted(project_dir.iterdir()):
    print(f"  {f.name}")


Project already exists at: D:\Workspace\Technologies\citi_dbt
Directory contents:
  .gitignore
  analyses
  dbt_project.yml
  logs
  macros
  models
  README.md
  seeds
  snapshots
  tests


`profiles.yml` holds the database connection configuration and lives in `~/.dbt/`, not inside the project directory.

In [3]:
from pathlib import Path

profiles_dir = Path.home() / ".dbt"
profiles_dir.mkdir(parents=True, exist_ok=True)

profiles_path = profiles_dir / "profiles.yml"
profiles_content = """citi_dbt:
  target: dev
  outputs:
    dev:
      type: postgres
      host: localhost
      port: 5432
      dbname: de_telemetry
      user: de_admin
      password: DeAdmin2026!
      schema: dbt_dev
      threads: 4
"""

profiles_path.write_text(profiles_content, encoding="utf-8")
print("profiles.yml written to ~/.dbt/profiles.yml")
print(profiles_path)

profiles.yml written to ~/.dbt/profiles.yml
C:\Users\shareuser\.dbt\profiles.yml


Verify dbt can reach Postgres before writing models.

In [4]:
DBT_PATH = "C:/py_venv/proj_educate/Scripts/dbt.exe"
import subprocess
from pathlib import Path

project_dir = Path(r"D:\Workspace\Technologies\citi_dbt")

result = subprocess.run(
    [DBT_PATH, "debug", "--project-dir", str(project_dir)],
    capture_output=True,
    text=True,
    cwd=str(project_dir.parent)
)

print(result.stdout)
if "All checks passed!" in result.stdout:
    print("dbt debug OK")
else:
    print(result.stderr)
    raise RuntimeError("dbt debug failed")

23:59:13  Running with dbt=1.11.7
23:59:13  dbt version: 1.11.7
23:59:13  python version: 3.12.9
23:59:13  python path: C:\py_venv\proj_educate\Scripts\python.exe
23:59:13  os info: Windows-11-10.0.26200-SP0
23:59:13  Using profiles dir at C:\Users\shareuser\.dbt
23:59:13  Using profiles.yml file at C:\Users\shareuser\.dbt\profiles.yml
23:59:13  Using dbt_project.yml file at D:\Workspace\Technologies\citi_dbt\dbt_project.yml
23:59:13  adapter type: postgres
23:59:13  adapter version: 1.10.0
23:59:13  Configuration:
23:59:13    profiles.yml file [OK found and valid]
23:59:13    dbt_project.yml file [OK found and valid]
23:59:13  Required dependencies:
23:59:13   - git [OK found]

23:59:13  Connection:
23:59:13    host: localhost
23:59:13    port: 5432
23:59:13    user: de_admin
23:59:13    database: de_telemetry
23:59:13    schema: dbt_dev
23:59:13    connect_timeout: 10
23:59:13    role: None
23:59:13    search_path: None
23:59:13    keepalives_idle: 0
23:59:13    sslmode: None
23:59:1

## Staging Model

Staging models are usually 1-to-1 with raw source tables. They handle light cleanup such as renaming, type casting, and normalization, but avoid business joins or heavy aggregations. The convention here is `models/staging/stg_alerts.sql`.

In [5]:
from pathlib import Path

project_dir = Path(r"D:\Workspace\Technologies\citi_dbt")
staging_dir = project_dir / "models" / "staging"
staging_dir.mkdir(parents=True, exist_ok=True)

stg_alerts_path = staging_dir / "stg_alerts.sql"
stg_alerts_sql = """-- stg_alerts: clean and rename raw alerts
SELECT
    alert_id,
    endpoint_id,
    UPPER(severity) AS severity,
    message,
    created_at::DATE AS alert_date,
    created_at AS alert_timestamp
FROM {{ source('de_telemetry', 'alerts') }}
"""

sources_yml_path = staging_dir / "sources.yml"
sources_yml = """version: 2
sources:
  - name: de_telemetry
    database: de_telemetry
    schema: public
    tables:
      - name: alerts
      - name: endpoints
"""

stg_alerts_path.write_text(stg_alerts_sql, encoding="utf-8")
sources_yml_path.write_text(sources_yml, encoding="utf-8")

print(f"Wrote: {stg_alerts_path}")
print(f"Wrote: {sources_yml_path}")

Wrote: D:\Workspace\Technologies\citi_dbt\models\staging\stg_alerts.sql
Wrote: D:\Workspace\Technologies\citi_dbt\models\staging\sources.yml


## Mart Model

Mart models contain business logic such as joins, aggregations, and analyst-facing structures. They are often materialized as tables for easy downstream querying.

In [6]:
from pathlib import Path

project_dir = Path(r"D:\Workspace\Technologies\citi_dbt")
marts_dir = project_dir / "models" / "marts"
marts_dir.mkdir(parents=True, exist_ok=True)

mart_path = marts_dir / "mart_alert_summary.sql"
mart_sql = """-- mart_alert_summary: daily alert counts per region and severity
{{ config(materialized='table') }}

WITH alerts AS (
    SELECT * FROM {{ ref('stg_alerts') }}
),
endpoints AS (
    SELECT endpoint_id, name, region, category
    FROM {{ source('de_telemetry', 'endpoints') }}
)
SELECT
    a.alert_date,
    e.region,
    a.severity,
    COUNT(*) AS alert_count
FROM alerts a
LEFT JOIN endpoints e USING (endpoint_id)
GROUP BY 1, 2, 3
ORDER BY 1 DESC, 4 DESC
"""

mart_path.write_text(mart_sql, encoding="utf-8")
print("mart_alert_summary.sql written")
print(mart_path)

mart_alert_summary.sql written
D:\Workspace\Technologies\citi_dbt\models\marts\mart_alert_summary.sql


`dbt run` compiles the SQL and executes it against Postgres.

In [7]:
DBT_PATH = "C:/py_venv/proj_educate/Scripts/dbt.exe"
import subprocess
from pathlib import Path

project_dir = Path(r"D:\Workspace\Technologies\citi_dbt")

result = subprocess.run(
    [DBT_PATH, "run", "--project-dir", str(project_dir)],
    capture_output=True,
    text=True,
    cwd=str(project_dir.parent)
)

print(result.stdout)
if "Completed successfully" in result.stdout:
    print("dbt run completed successfully")
else:
    print(result.stderr)

23:59:16  Running with dbt=1.11.7
23:59:17  Registered adapter: postgres=1.10.0
23:59:17  Unable to do partial parsing because saved manifest not found. Starting full parse.
23:59:18  Found 4 models, 4 data tests, 2 sources, 465 macros
23:59:18  
23:59:18  Concurrency: 4 threads (target='dev')
23:59:18  
23:59:19  1 of 4 START sql table model dbt_dev.my_first_dbt_model ........................ [RUN]
23:59:19  2 of 4 START sql view model dbt_dev.stg_alerts ................................. [RUN]
23:59:19  1 of 4 OK created sql table model dbt_dev.my_first_dbt_model ................... [SELECT 2 in 0.32s]
23:59:19  3 of 4 START sql view model dbt_dev.my_second_dbt_model ........................ [RUN]
23:59:19  2 of 4 OK created sql view model dbt_dev.stg_alerts ............................ [CREATE VIEW in 0.40s]
23:59:19  4 of 4 START sql table model dbt_dev.mart_alert_summary ........................ [RUN]
23:59:19  3 of 4 OK created sql view model dbt_dev.my_second_dbt_model ..........

## Testing

dbt generic tests let you assert quality rules like `not_null`, `unique`, `accepted_values`, and `relationships` directly in YAML.

In [8]:
DBT_PATH = "C:/py_venv/proj_educate/Scripts/dbt.exe"
from pathlib import Path
import subprocess

project_dir = Path(r"D:\Workspace\Technologies\citi_dbt")
staging_dir = project_dir / "models" / "staging"

stg_alerts_yml_path = staging_dir / "stg_alerts.yml"
stg_alerts_yml = """version: 2
models:
  - name: stg_alerts
    columns:
      - name: alert_id
        tests:
          - unique
          - not_null
      - name: severity
        tests:
          - not_null
          - accepted_values:
              values: ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
      - name: endpoint_id
        tests:
          - not_null
          - relationships:
              to: source('de_telemetry', 'endpoints')
              field: endpoint_id
"""

stg_alerts_yml_path.write_text(stg_alerts_yml, encoding="utf-8")
print(f"Wrote: {stg_alerts_yml_path}")

result = subprocess.run(
    [DBT_PATH, "test", "--project-dir", str(project_dir)],
    capture_output=True,
    text=True,
    cwd=str(project_dir.parent)
)

print(result.stdout)
print(result.stderr)

Wrote: D:\Workspace\Technologies\citi_dbt\models\staging\stg_alerts.yml


23:59:22  Running with dbt=1.11.7
23:59:23  Registered adapter: postgres=1.10.0
23:59:23  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'stg_alerts' in
package 'citi_dbt' (models\staging\stg_alerts.yml). Arguments to generic tests
should be nested under the `arguments` property.
23:59:23  Found 4 models, 10 data tests, 2 sources, 465 macros
23:59:23  
23:59:23  Concurrency: 4 threads (target='dev')
23:59:23  
23:59:23  1 of 10 START test accepted_values_stg_alerts_severity__LOW__MEDIUM__HIGH__CRITICAL  [RUN]
23:59:23  2 of 10 START test not_null_my_first_dbt_model_id .............................. [RUN]
23:59:23  3 of 10 START test not_null_my_second_dbt_model_id ............................. [RUN]
23:59:23  4 of 10 START test not_null_stg_alerts_alert_id ................................ [RUN]
23:59:24  3 of 10 PASS not_null_my_second_dbt_model_id ................................... [

Verify the mart in Postgres.

In [9]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="de_telemetry",
    user="de_admin",
    password="DeAdmin2026!"
)

try:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM dbt_dev.mart_alert_summary LIMIT 20")
        rows = cur.fetchall()
        for row in rows:
            print(row)
finally:
    conn.close()

(datetime.date(2026, 3, 23), 'NYC2', 'MEDIUM', 16)
(datetime.date(2026, 3, 23), 'NYC1', 'CRITICAL', 15)
(datetime.date(2026, 3, 23), 'SNG1', 'LOW', 15)
(datetime.date(2026, 3, 23), 'NYC1', 'MEDIUM', 13)
(datetime.date(2026, 3, 23), 'NYC1', 'LOW', 11)
(datetime.date(2026, 3, 23), 'NYC2', 'CRITICAL', 9)
(datetime.date(2026, 3, 23), 'SNG1', 'CRITICAL', 9)
(datetime.date(2026, 3, 23), 'NYC2', 'LOW', 9)
(datetime.date(2026, 3, 23), 'LON1', 'MEDIUM', 8)
(datetime.date(2026, 3, 23), 'SNG1', 'MEDIUM', 8)
(datetime.date(2026, 3, 23), 'NYC2', 'HIGH', 8)
(datetime.date(2026, 3, 23), 'NYC1', 'HIGH', 8)
(datetime.date(2026, 3, 23), 'SNG1', 'HIGH', 7)
(datetime.date(2026, 3, 23), 'LON1', 'LOW', 6)
(datetime.date(2026, 3, 23), 'LON1', 'CRITICAL', 6)
(datetime.date(2026, 3, 23), 'LON1', 'HIGH', 4)
(datetime.date(2026, 3, 22), 'SNG1', 'MEDIUM', 25)
(datetime.date(2026, 3, 22), 'NYC1', 'LOW', 23)
(datetime.date(2026, 3, 22), 'LON1', 'HIGH', 23)
(datetime.date(2026, 3, 22), 'LON1', 'MEDIUM', 23)


## What Just Happened

- Initialized a dbt project
- Wrote a staging model and a mart model
- Added `sources.yml` to define raw tables
- Added tests for uniqueness, null checks, accepted values, and relationships
- Ran `dbt run` and `dbt test`
- Queried the resulting mart in Postgres

These two models give ops a clean, tested, documented view of alerts by region — updated by an Airflow DAG calling `dbt run` nightly.

Next: run `dbt_concepts.md` for vocabulary, then move to Round 2 for incremental models and advanced patterns.